# PARC2026 — 74 M3 guarded 3-model smoke (A100)

72dとNotebook 73のpreflightを前提に、π0.5 / SmolVLA / OpenVLA-OFTを **64 samples / 2 optimizer updates** だけ実際に学習して、canonical sample order・checkpoint・計測を確認します。

`Run all` では学習を開始しません。下のforward / reverse各セルで明示的に `EXECUTE_... = True` にした方向だけ実行します。full M3 benchmarkはこのNotebookでは開始しません。


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_runner'
PIN = '51b1f1e71a53c559f46a688862f74c60c64048ff'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('74 smoke code:', got, flush=True)

gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
print('GPU:', gpu, flush=True)
if 'A100' not in gpu:
    raise RuntimeError('Notebook 74 requires an NVIDIA A100 runtime')

DRIVE = Path('/content/drive/MyDrive/parc2026-cache')
RUN_ROOT = DRIVE / 'model-benchmark-v1'
DATASET = DRIVE / 'datasets/lerobot_libero_plus_v3_train'
PLAN = RUN_ROOT / 'm3_training_adapter_preflight.json'
STREAMING = DRIVE / 'openvla-streaming-selected-v1/streaming_bridge_contract.json'
EXPECTED_HASH = '73ed0d3b0c5e73c745c0aa2e81517ce1fa40240c75f9eb040d65b6876ba08239'
manifest_candidates = [
    DRIVE / 'pi05-ablation-group-aware-v2/dataset_ablation_manifests_v2_group_aware/V2_SQRT_BALANCED_RAW.json',
    ROOT / 'outputs/dataset_ablation_manifests_v2_group_aware/V2_SQRT_BALANCED_RAW.json',
]
MANIFEST = None
for candidate in manifest_candidates:
    if not candidate.is_file():
        continue
    data = json.loads(candidate.read_text(encoding='utf-8'))
    if data.get('episode_ids_sha256') == EXPECTED_HASH:
        MANIFEST = candidate
        break
if MANIFEST is None:
    raise FileNotFoundError('Exact D10 manifest not found. Do not regenerate or reselect it.')
for required in (DATASET / 'meta/info.json', PLAN, STREAMING):
    if not required.is_file():
        raise FileNotFoundError(required)
plan = json.loads(PLAN.read_text(encoding='utf-8'))
if plan.get('status') != 'READY_FOR_GUARDED_SMOKE_IMPLEMENTATION':
    raise RuntimeError(f'Notebook 73 adapter plan is not ready: {plan.get("status")}')
if plan.get('selected_episode_ids_sha256') != EXPECTED_HASH or plan.get('training_started') is not False:
    raise RuntimeError('Notebook 73 adapter plan provenance/safety mismatch')
if int(plan.get('run_count', 0)) != 12:
    raise RuntimeError('Notebook 73 adapter plan must contain 12 run specs')
streaming = json.loads(STREAMING.read_text(encoding='utf-8'))
if streaming.get('status') != 'PASS' or streaming.get('bridge_type') != 'lerobot_streaming':
    raise RuntimeError('69c streaming contract is not PASS')
if streaming.get('source_episode_ids_sha256') != EXPECTED_HASH:
    raise RuntimeError('69c D10 hash mismatch')
if streaming.get('storage_policy', {}).get('full_rlds_materialized') is not False:
    raise RuntimeError('Notebook 74 refuses full RLDS materialization')
WORK_ROOT = ROOT
print(json.dumps({
    'status': 'READY_FOR_EXPLICIT_SMOKE',
    'manifest': str(MANIFEST),
    'dataset': str(DATASET),
    'adapter_plan': str(PLAN),
    'streaming_contract': str(STREAMING),
    'forward_order': plan['orders']['forward'],
    'reverse_order': plan['orders']['reverse'],
    'smoke_samples_per_model': 64,
    'smoke_optimizer_updates_per_model': 2,
    'benchmark_training_started': False,
}, indent=2), flush=True)


## Forward smoke
`EXECUTE_FORWARD = True` にした場合だけ `pi05 → smolvla → openvla_oft` の64-sample smokeを開始します。途中失敗時は停止し、PASS済みworkerは次回 `--resume-completed` で再利用します。


In [ ]:
EXECUTE_FORWARD = False  # smokeを開始するときだけ True に変更
if not EXECUTE_FORWARD:
    print('Forward smoke NOT started. Set EXECUTE_FORWARD=True explicitly to run it.', flush=True)
else:
    env = os.environ.copy()
    env['PARC_M3_EXECUTE'] = '1'
    subprocess.run([
        sys.executable, '-u', '-m', 'tools.benchmark.run_m3_guarded',
        '--repo-root', str(REPO), '--adapter-plan', str(PLAN),
        '--manifest', str(MANIFEST), '--dataset-root', str(DATASET),
        '--streaming-contract', str(STREAMING), '--work-root', str(WORK_ROOT),
        '--run-root', str(RUN_ROOT), '--mode', 'smoke', '--order', 'forward',
        '--track', 'equal_data', '--resume-completed',
    ], cwd=str(REPO), env=env, check=True)
    status_path = RUN_ROOT / 'smoke/forward/orchestrator_status.json'
    status = json.loads(status_path.read_text(encoding='utf-8'))
    if status.get('status') != 'PASS':
        raise RuntimeError(status)
    print('=== 74 FORWARD SMOKE: PASS ===', flush=True)
    print(json.dumps(status, indent=2), flush=True)


## Reverse smoke
Forwardとは独立です。`EXECUTE_REVERSE = True` にした場合だけ `openvla_oft → smolvla → pi05` を開始します。Forwardを実行しただけでReverseが自動開始されることはありません。


In [ ]:
EXECUTE_REVERSE = False  # smokeを開始するときだけ True に変更
if not EXECUTE_REVERSE:
    print('Reverse smoke NOT started. Set EXECUTE_REVERSE=True explicitly to run it.', flush=True)
else:
    env = os.environ.copy()
    env['PARC_M3_EXECUTE'] = '1'
    subprocess.run([
        sys.executable, '-u', '-m', 'tools.benchmark.run_m3_guarded',
        '--repo-root', str(REPO), '--adapter-plan', str(PLAN),
        '--manifest', str(MANIFEST), '--dataset-root', str(DATASET),
        '--streaming-contract', str(STREAMING), '--work-root', str(WORK_ROOT),
        '--run-root', str(RUN_ROOT), '--mode', 'smoke', '--order', 'reverse',
        '--track', 'equal_data', '--resume-completed',
    ], cwd=str(REPO), env=env, check=True)
    status_path = RUN_ROOT / 'smoke/reverse/orchestrator_status.json'
    status = json.loads(status_path.read_text(encoding='utf-8'))
    if status.get('status') != 'PASS':
        raise RuntimeError(status)
    print('=== 74 REVERSE SMOKE: PASS ===', flush=True)
    print(json.dumps(status, indent=2), flush=True)


## Smoke gate
両方向の3モデルがすべてPASSしたときだけ `m3_smoke_summary.json` を `READY_FOR_M3_BENCHMARK` にします。学習lossでモデル選定はしません。


In [ ]:
models = ['pi05', 'smolvla', 'openvla_oft']
expected_orders = {
    'forward': ['pi05', 'smolvla', 'openvla_oft'],
    'reverse': ['openvla_oft', 'smolvla', 'pi05'],
}
records = []
missing = []
for order in ('forward', 'reverse'):
    status_path = RUN_ROOT / f'smoke/{order}/orchestrator_status.json'
    if not status_path.is_file():
        missing.append(str(status_path))
        continue
    status = json.loads(status_path.read_text(encoding='utf-8'))
    if status.get('status') != 'PASS' or status.get('sequence') != expected_orders[order]:
        raise RuntimeError(f'{order} smoke orchestration is not PASS: {status}')
    for model in models:
        result_path = RUN_ROOT / f'smoke/{order}/{model}/training_result.json'
        if not result_path.is_file():
            missing.append(str(result_path))
            continue
        result = json.loads(result_path.read_text(encoding='utf-8'))
        if result.get('status') != 'PASS' or result.get('mode') != 'smoke':
            raise RuntimeError(result)
        if result.get('selected_episode_ids_sha256') != EXPECTED_HASH:
            raise RuntimeError(f'{model}/{order}: D10 hash mismatch')
        if int(result.get('samples_consumed', -1)) != 64:
            raise RuntimeError(f'{model}/{order}: smoke samples != 64')
        if int(result.get('optimizer_updates', -1)) != 2:
            raise RuntimeError(f'{model}/{order}: smoke optimizer updates != 2')
        if int(result.get('effective_batch_size', -1)) != 32:
            raise RuntimeError(f'{model}/{order}: effective batch != 32')
        evidence = json.loads(Path(result['training_evidence']).read_text(encoding='utf-8'))
        if evidence.get('canonical_sample_order_verified') is not True:
            raise RuntimeError(f'{model}/{order}: canonical sampling not verified')
        records.append(result)
if missing:
    print('Smoke gate not complete yet. Missing:', flush=True)
    for path in missing:
        print(' -', path, flush=True)
    print('No benchmark has been started.', flush=True)
else:
    for model in models:
        pair = [r for r in records if r['model'] == model]
        if len(pair) != 2:
            raise RuntimeError(f'{model}: expected forward+reverse smoke pair')
        if len({r['sampling_schedule_sha256'] for r in pair}) != 1:
            raise RuntimeError(f'{model}: forward/reverse schedule SHA differs')
    schedule_hashes = {r['sampling_schedule_sha256'] for r in records}
    if len(schedule_hashes) != 1:
        raise RuntimeError('3-model smoke did not use one common canonical schedule SHA')
    summary = {
        'schema_version': 1,
        'stage': 'M3_guarded_training_smoke',
        'status': 'READY_FOR_M3_BENCHMARK',
        'selected_dataset_variant': 'V2_SQRT_BALANCED_RAW',
        'selected_episode_ids_sha256': EXPECTED_HASH,
        'orders_passed': ['forward', 'reverse'],
        'models_passed': models,
        'sample_target_per_model_per_order': 64,
        'optimizer_updates_per_model_per_order': 2,
        'effective_batch_size': 32,
        'sampling_schedule_sha256': next(iter(schedule_hashes)),
        'benchmark_training_started': False,
        'training_loss_used_for_promotion': False,
    }
    out = RUN_ROOT / 'm3_smoke_summary.json'
    out.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(summary, indent=2), flush=True)
    print('=== 74 M3 GUARDED SMOKE: READY_FOR_M3_BENCHMARK ===', flush=True)
    print('Full benchmark has NOT started.', flush=True)
